This is a trial coding for initiating REST API with Hadoop cluster. Please refer main.ipynb for the main submission!

In [4]:
from starbase import Connection

# 1. Connect to the HBase REST server port we started inside the VM
# (127.0.0.1 works perfectly because your port 8000 is mapped via the sandbox)
c = Connection("127.0.0.1", "8000")

ratings = c.table('ratings')

# 2. Clean up any existing tables from previous attempts
if (ratings.exists()):
    print("Dropping existing ratings table\n")
    ratings.drop()

# 3. Create the 'rating' column family
ratings.create('rating')
print("Parsing the ml-100k ratings data... \n")

# 4. OPEN THE DATA FROM YOUR EXACT WINDOWS PATH
ratingFile = open(r"C:\Users\PC 14\Desktop\github - muz\P166246_Assg2_STQD6324\ml-100k\u.data", "r")

batch = ratings.batch()

# 5. Parse and add rows to the batch array
for line in ratingFile:
    (userID, movieID, rating, timestamp) = line.strip().split('\t')
    batch.update(userID, {'rating': {movieID: rating}})

ratingFile.close()

# 6. Commit via REST API gateway to the Linux Sandbox
print("Committing ratings data to HBase via REST gateway...\n")
batch.commit(finalize=True)

# 7. Fetch data DIRECTLY BACK OUT OF HBASE to verify it works!
print("Get back ratings for some users...\n")
print("Ratings for user ID 35:\n")
print(ratings.fetch("35"))

Dropping existing ratings table

Parsing the ml-100k ratings data... 

Committing ratings data to HBase via REST gateway...

Get back ratings for some users...

Ratings for user ID 35:

{'rating': {'1025': '3', '242': '2', '243': '2', '258': '2', '259': '4', '261': '3', '264': '2', '266': '3', '300': '5', '321': '3', '322': '3', '326': '3', '327': '3', '328': '3', '332': '4', '333': '4', '358': '1', '678': '3', '680': '4', '748': '4', '876': '2', '877': '2', '879': '4', '881': '2', '937': '4'}}


In [7]:
from starbase import Connection

# =========================================================================
# 1. INITIALIZE GLOBAL HBASE REST CONNECTION
# =========================================================================
# Connect to your Sandbox HBase REST gateway (running inside your VM)
c = Connection("127.0.0.1", "8000")
table_name = 'movielens_recommendations'
table = c.table(table_name)


# =========================================================================
# 2. DEFINE THE REUSABLE LOOKUP FUNCTION
# =========================================================================
def get_recommendations(user_input):
    """
    Fetches and cleanly prints movie recommendations directly from HBase.
    Accepts a single integer (e.g., 50) or a list of integers (e.g., [1, 50, 100]).
    """
    # Verify the table exists before attempting queries
    if not table.exists():
        print("Error: Table '{}' not found in HBase! Ensure your Spark script ran successfully.".format(table_name))
        return

    # Convert a single user ID into a list so our loop handles it uniformly
    user_ids = user_input if isinstance(user_input, list) else [user_input]
    
    for user_id in user_ids:
        # Format the row key to match the padded 6-digit schema (e.g., user_000050)
        row_key = "user_{:06d}".format(user_id)
        
        print("=" * 65)
        print("HBASE LOOKUP VIA REST — User {}".format(user_id))
        print("HBase Row Key: {}".format(row_key))
        print("=" * 65)
        
        # Pull wide-column record straight from HBase memory
        row_data = table.fetch(row_key)
        
        if not row_data:
            print("  No recommendation profile found in HBase for row key: {}".format(row_key))
        else:
            # Extract the nested column family dictionary
            recommendations = row_data.get('recommendations', {})
            
            print("  {:<5} {:<10} {:<42} {}".format("Rank", "Movie ID", "Movie Title", "Avg Rating"))
            print("  " + "-" * 61)
            
            # Print the top-5 elements stored in HBase wide columns
            for rank in range(1, 6):
                title_col = 'rank_{}_title'.format(rank)
                id_col    = 'rank_{}_movie_id'.format(rank)
                rate_col  = 'rank_{}_avg_rating'.format(rank)
                
                title = recommendations.get(title_col, "Unknown Title")
                m_id  = recommendations.get(id_col, "N/A")
                rate  = recommendations.get(rate_col, "N/A")
                
                print("  {:<5} {:<10} {:<42} {}".format(rank, m_id, title[:41], rate))
                
        print("=" * 65 + "\n")

print("Function 'get_recommendations' successfully defined and ready to use!")

Function 'get_recommendations' successfully defined and ready to use!


In [8]:
get_recommendations(35)

HBASE LOOKUP VIA REST — User 35
HBase Row Key: user_000035
  Rank  Movie ID   Movie Title                                Avg Rating
  -------------------------------------------------------------
  1     50         Star Wars (1977)                           4.3596
  2     127        Godfather, The (1972)                      4.2833
  3     174        Raiders of the Lost Ark (1981)             4.2524
  4     313        Titanic (1997)                             4.2457
  5     172        Empire Strikes Back, The (1980)            4.2065



In [9]:
get_recommendations(89)

HBASE LOOKUP VIA REST — User 89
HBase Row Key: user_000089
  Rank  Movie ID   Movie Title                                Avg Rating
  -------------------------------------------------------------
  1     1449       Pather Panchali (1955)                     4.6250
  2     318        Schindler's List (1993)                    4.4664
  3     483        Casablanca (1942)                          4.4568
  4     64         Shawshank Redemption, The (1994)           4.4452
  5     178        12 Angry Men (1957)                        4.3440



In [16]:
get_recommendations(0) #user 1 is numbered 0

HBASE LOOKUP VIA REST — User 0
HBase Row Key: user_000000
  Rank  Movie ID   Movie Title                                Avg Rating
  -------------------------------------------------------------
  1     318        Schindler's List (1993)                    4.4664
  2     483        Casablanca (1942)                          4.4568
  3     474        Dr. Strangelove or: How I Learned to Stop  4.2526
  4     511        Lawrence of Arabia (1962)                  4.2312
  5     641        Paths of Glory (1957)                      4.2121



In [17]:
get_recommendations(943) #user 944 is numbered 943

HBASE LOOKUP VIA REST — User 943
HBase Row Key: user_000943
  Rank  Movie ID   Movie Title                                Avg Rating
  -------------------------------------------------------------
  1     313        Titanic (1997)                             4.2457
  2     515        Boot, Das (1981)                           4.2040
  3     498        African Queen, The (1951)                  4.1842
  4     651        Glory (1989)                               4.0760
  5     183        Alien (1979)                               4.0344

